# Test of G2 (`02_repository_issue_flow`)

This notebook tests `02_repository_issue_flow` with a small hand-made Silver table in the schema `github_lakehouse.test_gold_g2`. It runs the real notebook with `%run` and `test_schema = test_gold_g2`, so **production tables are never written**. The last check section proves that.

**Before you run it:** `08_gold_audit_table` must have run once, and the Silver table `silver.issue_events` must exist. The test copies the definitions of both tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: an empty Silver table. T14, T15: a source date without rows. Both give `NO_INPUT` and leave Gold unchanged |
| Schema | T7: the Gold table has exactly the columns and types of the contract |
| Exact values | T7: hand-made rows give the hand-calculated Gold rows |
| Missing values | T7: a close event without usable times, an unknown action, an issue without a key, an event without a name, an event without a date |
| Zero base | T7: a day without an eligible close event has NULL for the average and the median, not 0 |
| Rerun | T8, T9: a second run changes nothing |
| Late event | T10, T11: new Silver rows change the right Gold rows, and the other days are not touched |
| Removed event | T12, T13: removed Silver rows lower one Gold row and remove another |
| Failed check | T16 to T18: a failing quality gate leaves Gold unchanged, then a recovery run |
| A day that is gone | T19, T20: a run for all dates removes a day that no longer exists in Silver |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The Silver test table is an empty copy of the real `silver.issue_events`, and the audit test table is an empty copy of the real audit table.

The Gold test table is not created here. The notebook under test creates it, which also tests its `CREATE TABLE` statement.

In [0]:
TEST_SCHEMA = "test_gold_g2"

TEST_SILVER = f"github_lakehouse.{TEST_SCHEMA}.issue_events"
TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.repository_issue_flow"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.repository_issue_flow_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

PROD_SILVER = "github_lakehouse.silver.issue_events"
PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_GOLD = "github_lakehouse.gold.repository_issue_flow"
PROD_STAGING = "github_lakehouse.gold_staging.repository_issue_flow"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

if not spark.catalog.tableExists(PROD_SILVER):
    raise RuntimeError(f"{PROD_SILVER} does not exist. Run the Silver notebook 03_issue_events first.")

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE {PROD_SILVER}")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version  :", prod_audit_version_before)
print("Production Gold version   :", prod_gold_version_before)
print("Production staging version:", prod_staging_version_before)

## T3. Run on the empty Silver table

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

An empty Silver table has no days to rebuild. The run must end as `NO_INPUT`, publish nothing and still write its audit row.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test Silver table' AS check_name,
               '{SILVER_TABLE}' AS actual, '{TEST_SILVER}' AS expected
        UNION ALL
        SELECT '02 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '03 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '04 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '07 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '08 audit pipeline_name', MAX(pipeline_name), 'gold_repository_issue_flow'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '09 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '10 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '11 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '12 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '13 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '14 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '15 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Silver rows

25 issue events on three days. Only the columns that G2 reads are filled, the others stay NULL.

**Repository 100 on 1 February: 13 events, one for every rule**

| Event | Action | Issue | What it tests |
|---|---|---|---|
| ...01 | `opened` | 1 | An issue is opened |
| ...02 | `closed` | 1 | and closed the same day, 2 hours after its creation |
| ...03 | `reopened` | 1 | then reopened |
| ...04 | `closed` | 1 | and closed again, 5 hours after its **original** creation |
| ...05 | `closed` | 2 | A close event without a close time: not eligible |
| ...06 | `opened` | 3 | A close time on an `opened` event: not eligible |
| ...07 | `closed` | 4 | Closed before it was created: not eligible |
| ...08 | `closed` | 5 | No creation time: not eligible |
| ...09 | none | 6 | An unknown action |
| ...10 | `labeled` | 6 | A known action that is none of the three |
| ...11 | `opened` | number 77 | No `issue_id`: the issue is identified by its number |
| ...12 | `closed` | number 77 | The same issue, closed after 20 minutes. Same second as ...11, but the larger `event_id` |
| ...13 | `opened` | none | No ID and no number, and no repository name. It is the latest event of the day |

**The other rows**

| Repository | Day | Events | What it tests |
|---|---|---|---|
| 200 | 1 Feb | 2 | A day with only `opened` events: nothing to average |
| 200 | none | 1 | An event without event time and date: it cannot be placed on a day |
| 200 | 2 Feb | 1 | The same repository on a second day, with a new name. Closed after 24 hours |
| 400 | 2 Feb | 1 | A repository whose events have no name |
| 300 | 5 Feb | 4 | Four issues closed after 1, 2, 3 and 10 hours: the median of an even number of values |
| 310 | 5 Feb | 1 | Created 1 December 2023 12:00, closed 5 February 2024 12:00: 1,584 hours. The event itself is 30 minutes later than the close time, and that must not count |
| 320 | 5 Feb | 2 | Closed after 20 and 60 minutes: an average and a median that must be rounded |

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     issue_action, issue_id, issue_number, issue_created_at, issue_closed_at, _source_date)
VALUES
    -- Repository 100 on 1 February
    ('35000000001', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', 100, 'org/alpha',     'opened',   1,    1,    TIMESTAMP'2024-02-01 08:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000002', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', 100, 'org/alpha',     'closed',   1,    1,    TIMESTAMP'2024-02-01 08:00:00', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01'),
    ('35000000003', TIMESTAMP'2024-02-01 11:00:00', DATE'2024-02-01', 100, 'org/alpha',     'reopened', 1,    1,    TIMESTAMP'2024-02-01 08:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000004', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 100, 'org/alpha',     'closed',   1,    1,    TIMESTAMP'2024-02-01 08:00:00', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01'),
    ('35000000005', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 100, 'org/alpha',     'closed',   2,    2,    TIMESTAMP'2024-01-31 12:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000006', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01', 100, 'org/alpha',     'opened',   3,    3,    TIMESTAMP'2024-02-01 15:00:00', TIMESTAMP'2024-02-01 15:30:00', DATE'2024-02-01'),
    ('35000000007', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 100, 'org/alpha',     'closed',   4,    4,    TIMESTAMP'2024-02-01 16:00:00', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01'),
    ('35000000008', TIMESTAMP'2024-02-01 17:00:00', DATE'2024-02-01', 100, 'org/alpha',     'closed',   5,    5,    NULL,                           TIMESTAMP'2024-02-01 17:00:00', DATE'2024-02-01'),
    ('35000000009', TIMESTAMP'2024-02-01 18:00:00', DATE'2024-02-01', 100, 'org/alpha',     NULL,       6,    6,    TIMESTAMP'2024-02-01 18:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000010', TIMESTAMP'2024-02-01 18:30:00', DATE'2024-02-01', 100, 'org/alpha',     'labeled',  6,    6,    TIMESTAMP'2024-02-01 18:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000011', TIMESTAMP'2024-02-01 19:20:00', DATE'2024-02-01', 100, 'org/alpha-old', 'opened',   NULL, 77,   TIMESTAMP'2024-02-01 19:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000012', TIMESTAMP'2024-02-01 19:20:00', DATE'2024-02-01', 100, 'org/alpha-new', 'closed',   NULL, 77,   TIMESTAMP'2024-02-01 19:00:00', TIMESTAMP'2024-02-01 19:20:00', DATE'2024-02-01'),
    ('35000000013', TIMESTAMP'2024-02-01 20:00:00', DATE'2024-02-01', 100, NULL,            'opened',   NULL, NULL, TIMESTAMP'2024-02-01 20:00:00', NULL,                           DATE'2024-02-01'),

    -- Repository 200 on 1 February: only opened events
    ('35000000014', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 200, 'org/beta',      'opened',   20,   1,    TIMESTAMP'2024-02-01 09:00:00', NULL,                           DATE'2024-02-01'),
    ('35000000015', TIMESTAMP'2024-02-01 09:30:00', DATE'2024-02-01', 200, 'org/beta',      'opened',   21,   2,    TIMESTAMP'2024-02-01 09:30:00', NULL,                           DATE'2024-02-01'),

    -- Repository 200: an event without event time and event date
    ('35000000016', NULL,                           NULL,             200, 'org/beta',      'opened',   23,   4,    TIMESTAMP'2024-02-01 10:00:00', NULL,                           DATE'2024-02-01'),

    -- Repository 200 on 2 February: a new name, closed after 24 hours
    ('35000000017', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 200, 'org/beta-renamed', 'closed', 20,  1,    TIMESTAMP'2024-02-01 09:00:00', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02'),

    -- Repository 400 on 2 February: no name
    ('35000000018', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 400, NULL,            'opened',   40,   1,    TIMESTAMP'2024-02-02 10:00:00', NULL,                           DATE'2024-02-02'),

    -- Repository 300 on 5 February: closed after 1, 2, 3 and 10 hours
    ('35000000019', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 300, 'org/gamma',     'closed',   30,   1,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05'),
    ('35000000020', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 300, 'org/gamma',     'closed',   31,   2,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05'),
    ('35000000021', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 300, 'org/gamma',     'closed',   32,   3,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05'),
    ('35000000022', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 300, 'org/gamma',     'closed',   33,   4,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05'),

    -- Repository 310 on 5 February: an issue from 1 December 2023. The event is 30 minutes after the close time
    ('35000000023', TIMESTAMP'2024-02-05 12:30:00', DATE'2024-02-05', 310, 'org/delta',     'closed',   50,   9,    TIMESTAMP'2023-12-01 12:00:00', TIMESTAMP'2024-02-05 12:00:00', DATE'2024-02-05'),

    -- Repository 320 on 5 February: closed after 20 and 60 minutes
    ('35000000024', TIMESTAMP'2024-02-05 06:20:00', DATE'2024-02-05', 320, 'org/epsilon',   'closed',   60,   1,    TIMESTAMP'2024-02-05 06:00:00', TIMESTAMP'2024-02-05 06:20:00', DATE'2024-02-05'),
    ('35000000025', TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05', 320, 'org/epsilon',   'closed',   61,   2,    TIMESTAMP'2024-02-05 07:00:00', TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

### A one-line summary of every Gold row

The checks below compare whole Gold rows. This view writes the twelve value columns of a row into one readable line, for example:

`events=2 known=2 opened=2 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=2 issues=2 name=org/beta`

The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        activity_date,
        concat(
            'events=',    CAST(issue_event_count AS STRING),
            ' known=',    CAST(issue_action_known_rows AS STRING),
            ' opened=',   CAST(issue_opened_actions AS STRING),
            ' closed=',   CAST(issue_closed_actions AS STRING),
            ' reopened=', CAST(issue_reopened_actions AS STRING),
            ' other=',    CAST(issue_other_action_rows AS STRING),
            ' eligible=', CAST(issue_close_duration_eligible_rows AS STRING),
            ' avg=',      COALESCE(CAST(avg_resolution_hours AS STRING), 'NULL'),
            ' p50=',      COALESCE(CAST(p50_resolution_hours AS STRING), 'NULL'),
            ' usable=',   CAST(issues_with_usable_key_rows AS STRING),
            ' issues=',   CAST(distinct_issues_observed AS STRING),
            ' name=',     COALESCE(repo_name, 'NULL')
        ) AS summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T6. Run 1: all dates

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="" $run_id="test-run1"

## T7. Checks of run 1: schema and exact values

Worked out by hand.

**The run.** 24 of the 25 rows have an event date, on 3 days. The row without a date is left out and counted. The 24 rows belong to 7 repository-days.

**Repository 100 on 1 February.**

- 13 events. 12 have a known action: 4 opened (...01, ...06, ...11, ...13), 6 closed (...02, ...04, ...05, ...07, ...08, ...12), 1 reopened, 1 other.
- Of the 6 close events, 3 are eligible: 2 hours, 5 hours and 20 minutes (0.3333 hours). The other 3 have no close time, a close time before the creation, or no creation time.
- Average: (2 + 5 + 0.3333) ÷ 3 = 2.4444, rounded 2.44. Median of 0.3333, 2 and 5: the middle value, 2.
- 12 events have a usable issue key, all but ...13. They belong to 7 issues: the IDs 1, 2, 3, 4, 5 and 6, and the number 77.
- Label: the latest event (...13) has no name, so it is skipped. The next two share the same second. The larger `event_id` is ...12, so the label is `org/alpha-new`.

**The other rows.**

- Repository 200 on 1 February: 2 opened events, nothing closed. The average and the median are NULL and the eligible count is 0.
- Repository 200 on 2 February: one close after 24 hours. Its label is the new name.
- Repository 400 on 2 February: one event without a name. The row exists and its label is NULL.
- Repository 300 on 5 February: 1, 2, 3 and 10 hours. Average 16 ÷ 4 = 4. Median: the average of the two middle values, (2 + 3) ÷ 2 = 2.5.
- Repository 310 on 5 February: 1 December 12:00 to 5 February 12:00 is 30 + 31 + 5 = 66 days, which is 1,584 hours. It is counted on 5 February, the day of the close event. The hours end at the close time, not at the time of the event, so the 30 minutes between the two are not added.
- Repository 320 on 5 February: 0.3333 and 1 hour. Average and median are both 0.6667, rounded 0.67.

**The totals of the run.** 24 events, 23 with a known action: 7 opened, 14 closed, 1 reopened, 1 other. 11 eligible close events. 23 events with a usable key.

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

expected_schema = (
    "struct<repo_id:bigint,activity_date:date,repo_name:string,"
    "issue_event_count:bigint,issue_action_known_rows:bigint,issue_opened_actions:bigint,"
    "issue_closed_actions:bigint,issue_reopened_actions:bigint,issue_other_action_rows:bigint,"
    "issue_close_duration_eligible_rows:bigint,avg_resolution_hours:double,p50_resolution_hours:double,"
    "issues_with_usable_key_rows:bigint,distinct_issues_observed:bigint,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", expected_schema)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == expected_schema:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == expected_schema:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 different days', CAST(COUNT(DISTINCT activity_date) AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 sum of issue_event_count', CAST(SUM(issue_event_count) AS STRING), '24' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 100 on 1 February', MAX(summary), 'events=13 known=12 opened=4 closed=6 reopened=1 other=1 eligible=3 avg=2.44 p50=2.0 usable=12 issues=7 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '08 repository 200 on 1 February (only opened)', MAX(summary), 'events=2 known=2 opened=2 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=2 issues=2 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 repository 200 on 2 February (new name)', MAX(summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible=1 avg=24.0 p50=24.0 usable=1 issues=1 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '10 repository 400 on 2 February (no name)', MAX(summary), 'events=1 known=1 opened=1 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=1 issues=1 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '11 repository 300 on 5 February (median of 4 values)', MAX(summary), 'events=4 known=4 opened=0 closed=4 reopened=0 other=0 eligible=4 avg=4.0 p50=2.5 usable=4 issues=4 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '12 repository 310 on 5 February (1,584 hours)', MAX(summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible=1 avg=1584.0 p50=1584.0 usable=1 issues=1 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '13 repository 320 on 5 February (rounding)', MAX(summary), 'events=2 known=2 opened=0 closed=2 reopened=0 other=0 eligible=2 avg=0.67 p50=0.67 usable=2 issues=2 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 320 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '14 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '15 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '16 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '17 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '18 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '24'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '19 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '20 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '7'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '21 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '22 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '23 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '24 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '25 audit pipeline_name', MAX(pipeline_name), 'gold_repository_issue_flow'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '26 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '27 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '28 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['issue_event_count'] AS STRING), ' known=', CAST(quality_metrics['issue_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['issue_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['issue_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['issue_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['issue_other_action_rows'] AS STRING), ' eligible=', CAST(quality_metrics['issue_close_duration_eligible_rows'] AS STRING), ' usable=', CAST(quality_metrics['issues_with_usable_key_rows'] AS STRING))),
               'repository_days=7 events=24 known=23 opened=7 closed=14 reopened=1 other=1 eligible=11 usable=23'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the one-line summary of every row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, activity_date, summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

A run for all dates replaces the whole table, so every row now carries the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, summary FROM gold_summary
            EXCEPT
            SELECT repo_id, activity_date, summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '04 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, activity_date, summary FROM gold_summary
        )
        UNION ALL
        SELECT '05 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '7'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['issue_event_count'] AS STRING), ' known=', CAST(quality_metrics['issue_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['issue_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['issue_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['issue_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['issue_other_action_rows'] AS STRING), ' eligible=', CAST(quality_metrics['issue_close_duration_eligible_rows'] AS STRING), ' usable=', CAST(quality_metrics['issues_with_usable_key_rows'] AS STRING))),
               'repository_days=7 events=24 known=23 opened=7 closed=14 reopened=1 other=1 eligible=11 usable=23'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '08 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events, for one source date

Three new Silver rows arrive with the source date 2 February:

| Event | Repository | Event date | What it tests |
|---|---|---|---|
| ...26 | 200 | 2 Feb | A second close on a day that already has a Gold row: 26 hours |
| ...27 | 500 | 2 Feb | A repository that had no Gold row yet |
| ...28 | 200 | **1 Feb** | A late event: it was loaded with the source date 2 February, but it happened on 1 February |

The run gets `source_date = 2024-02-02`.

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     issue_action, issue_id, issue_number, issue_created_at, issue_closed_at, _source_date)
VALUES
    ('35000000026', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 200, 'org/beta-renamed', 'closed', 21, 2, TIMESTAMP'2024-02-01 09:30:00', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02'),
    ('35000000027', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 500, 'org/zeta',         'opened', 70, 1, TIMESTAMP'2024-02-02 12:00:00', NULL,                           DATE'2024-02-02'),
    ('35000000028', TIMESTAMP'2024-02-01 23:59:00', DATE'2024-02-01', 200, 'org/beta',         'opened', 22, 3, TIMESTAMP'2024-02-01 23:59:00', NULL,                           DATE'2024-02-02')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-02-02" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- The source date 2 February has 5 rows now: ...17, ...18, ...26, ...27 and ...28. Their event dates are 1 and 2 February, so **two** days are affected. The row without a date belongs to the source date 1 February and is not in the scope of this run.
- Both days are rebuilt from **all** their rows: 16 rows on 1 February (13 + 2 + the late one) and 4 rows on 2 February. 20 rows, 5 repository-days.
- Repository 200 on 1 February has 3 opened events now.
- Repository 200 on 2 February has 2 closes: 24 and 26 hours. Average and median are 25.
- Repository 500 on 2 February is new.
- Repository 100 on 1 February has the same numbers as before, but it was written again by this run.
- The three rows of 5 February are not touched. They still carry the ID of run 2.
- Gold has 5 + 3 = 8 rows.
- Totals of the run: 20 events, 19 with a known action: 9 opened, 8 closed, 1 reopened, 1 other. 5 eligible close events. 19 with a usable key.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '20'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '5'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '08 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 3', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run3') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 repository 200 on 1 February (late event)', MAX(summary), 'events=3 known=3 opened=3 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=3 issues=3 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 200 on 2 February (second close)', MAX(summary), 'events=2 known=2 opened=0 closed=2 reopened=0 other=0 eligible=2 avg=25.0 p50=25.0 usable=2 issues=2 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '13 repository 500 on 2 February (new row)', MAX(summary), 'events=1 known=1 opened=1 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=1 issues=1 name=org/zeta'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '14 repository 100 on 1 February (same numbers)', MAX(summary), 'events=13 known=12 opened=4 closed=6 reopened=1 other=1 eligible=3 avg=2.44 p50=2.0 usable=12 issues=7 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '15 repository 400 on 2 February (same numbers)', MAX(summary), 'events=1 known=1 opened=1 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=1 issues=1 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 300 on 5 February (not touched)', MAX(summary), 'events=4 known=4 opened=0 closed=4 reopened=0 other=0 eligible=4 avg=4.0 p50=2.5 usable=4 issues=4 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '17 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['issue_event_count'] AS STRING), ' known=', CAST(quality_metrics['issue_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['issue_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['issue_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['issue_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['issue_other_action_rows'] AS STRING), ' eligible=', CAST(quality_metrics['issue_close_duration_eligible_rows'] AS STRING), ' usable=', CAST(quality_metrics['issues_with_usable_key_rows'] AS STRING))),
               'repository_days=5 events=20 known=19 opened=9 closed=8 reopened=1 other=1 eligible=5 usable=19'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '18 different keys in Gold', CAST(COUNT(DISTINCT repo_id, activity_date) AS STRING), '8' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Two Silver rows are removed: the second close of repository 200 on 2 February (...26) and the only event of repository 500 (...27). Then the source date 2 February runs again.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_id IN ('35000000026', '35000000027')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-02-02" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- The source date 2 February has 3 rows left: ...17, ...18 and the late event ...28. The affected days are still 1 and 2 February.
- 16 rows on 1 February and 2 rows on 2 February: 18 rows, 4 repository-days.
- Repository 200 on 2 February is back to one close after 24 hours.
- Repository 500 has no event any more, so its Gold row is gone.
- Gold has 4 + 3 = 7 rows. The rows of 5 February still carry the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '03 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '04 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of repository 500 (removed)', CAST(count_if(repo_id = 500) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 200 on 2 February (lowered)', MAX(summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible=1 avg=24.0 p50=24.0 usable=1 issues=1 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '08 repository 200 on 1 February (late event still there)', MAX(summary), 'events=3 known=3 opened=3 closed=0 reopened=0 other=0 eligible=0 avg=NULL p50=NULL usable=3 issues=3 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['issue_event_count'] AS STRING), ' known=', CAST(quality_metrics['issue_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['issue_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['issue_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['issue_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['issue_other_action_rows'] AS STRING), ' eligible=', CAST(quality_metrics['issue_close_duration_eligible_rows'] AS STRING), ' usable=', CAST(quality_metrics['issues_with_usable_key_rows'] AS STRING))),
               'repository_days=4 events=18 known=17 opened=8 closed=7 reopened=1 other=1 eligible=4 usable=17'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: a source date without rows

No Silver row has the source date 3 February. The run gets `source_date = 2024-02-03`.

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-02-03" $run_id="test-run5"

## T15. Checks of run 5: `NO_INPUT`, Gold unchanged

There is no affected day, so nothing is published. The Gold table still holds the 4 rows of run 4 and the 3 rows of run 2. The run still writes its audit row.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '05 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 Gold rows still from run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 Gold rows still from run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '3' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: a quality gate fails

The Silver notebook never lets a row without `repo_id` into Silver. The test puts one such row into the test table anyway, on 5 February. Its group has no repository, so the Gold key would be NULL. The gate `KEY_IS_NULL` must catch it.

The cell below prepares the fault. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     issue_action, issue_id, issue_number, issue_created_at, issue_closed_at, _source_date)
VALUES
    ('35000000029', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', NULL, 'org/broken', 'opened', 80, 1, TIMESTAMP'2024-02-05 13:00:00', NULL, DATE'2024-02-05')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: repository_issue_flow FAILED: Quality gates failed: KEY_IS_NULL`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-02-05" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- The source date 5 February has 8 rows: the 7 good ones and the bad one. The affected day is 5 February.
- The run staged 4 rows: repositories 300, 310 and 320, and one row without a repository.
- It must have written its audit row with the status `FAILED` and the name of the gate.
- The Gold table must be exactly what it was: 7 rows, and the rows of 5 February still carry the ID of run 2. The rejected result is only in the staging table.

After the checks, the test removes the bad row, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '08 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows written by the failed run', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '4' FROM {TEST_STAGING}
        UNION ALL
        SELECT '13 staging rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '1' FROM {TEST_STAGING}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_id = '35000000029'
""")

print("Silver test rows after the repair:", spark.table(TEST_SILVER).count())

## T18. Run 7: recovery

The same run again, now without the bad row. The three rows of 5 February are replaced. The four rows of 1 and 2 February are not touched and still carry the ID of run 4.

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-02-05" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '3'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of 5 February written by run 7', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run7') AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of 1 and 2 February still from run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 310 on 5 February', MAX(summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible=1 avg=1584.0 p50=1584.0 usable=1 issues=1 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '09 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: a day that no longer exists in Silver

All Silver rows of 5 February are removed. A run for the source date 5 February would now find no rows and end as `NO_INPUT`, as in T15, and the three Gold rows of that day would stay. The way to remove such a day is a run for all dates: it replaces the whole table.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_date = DATE'2024-02-05'
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: Silver has 19 rows left. 18 have an event date, on 1 and 2 February, and 1 has none. Gold has the 4 rows of those two days and no row of 5 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', CAST(MAX(source_rows_read['issue_events']) AS STRING), '18'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '05 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows of 5 February', CAST(count_if(activity_date = DATE'2024-02-05') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../02_repository_issue_flow $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_g2" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version  :", prod_audit_version_after)
print("Production Gold version   :", prod_gold_version_after)
print("Production staging version:", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=2'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '09 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '10 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_repository_issue_flow' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)